# Providers 03 - vLLM Runtime

Objetivo: consumir un endpoint vLLM OpenAI-compatible mediante `vllm-runtime`. Instalar CUDA, descargar modelos y administrar procesos son responsabilidades operativas y quedan fuera del notebook canonico.

**Lugar en el modelo:** vLLM es el Provider de un endpoint OpenAI-compatible; Agentic Systems no administra el servidor.

**Evidencia exigida:** con endpoint y modelo válidos, la ejecución debe producir un `RunResult` exitoso con `engine="vllm-runtime"`; sin ellos debe reportar `not-run` con motivo.

**Límite de la evidencia:** detectar variables de entorno no prueba conectividad ni compatibilidad del modelo; la ejecución live sí.

## Parametros de la demostracion

| Variable | Default | Proposito |
|---|---|---|
| RUN_VLLM_LIVE | 1 | Usa 0 para desactivar la llamada real. |
| VLLM_BASE_URL | provider default | Override del endpoint OpenAI-compatible. |
| VLLM_MODEL | provider default | Override del identificador servido por vLLM. |
| VLLM_API_KEY | opcional | Token cuando el endpoint lo requiere. |

## Contrato de la demostracion

El notebook prueba el provider Unicamente a traves de la fachada publica:

```text
toolkit.runtime - toolkit.system - system.agent - RunResult
```

La celda live se habilita con una variable explicita. Sin credenciales o endpoint, el notebook permanece ejecutable y muestra un estado not-run estructurado. Cuando la variable esta activa, cualquier error real del provider debe ser visible.

In [ ]:
import os

import agentic_systems as toolkit

vllm_environment = toolkit.vllm_environment_snapshot()

RUN_VLLM_LIVE = os.getenv("RUN_VLLM_LIVE", "1").strip().lower() in {"1", "true", "yes"}
AGENT_NAME = "vllm_public_api_probe"

toolkit.show_json({
    "package": toolkit.__name__,
    "version": toolkit.__version__,
    "run_live": RUN_VLLM_LIVE,
    "environment": vllm_environment,
}, title="Preflight vLLM")

## 1) Declarar runtime y limites

Configura `VLLM_BASE_URL`, `VLLM_MODEL` y, si aplica, `VLLM_API_KEY` antes de iniciar el kernel. El runtime no administra el servidor.

In [ ]:
scheduler = toolkit.scheduler(
    timeout_s=90,
    max_retries=0,
    max_tool_calls=1,
    max_turns=3,
    max_concurrency=1,
)

runtime = toolkit.runtime(
    provider="vllm-runtime",
    model=os.getenv("VLLM_MODEL"),
    scheduler=scheduler,
    metadata={"tutorial": "providers/vllm"},
)

toolkit.show_json(runtime.describe(), title="vLLM RuntimeConfig")

## 2) Crear system, tool y agent

La definicion es deliberadamente identica a OpenAI y Bedrock. La unica diferencia es el `RuntimeConfig`.

In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    """Verifica un simbolo contra la superficie publica instalada."""
    return {
        "symbol": symbol,
        "is_public": symbol in toolkit.__all__,
        "package_version": toolkit.__version__,
    }

system = toolkit.system(runtime=runtime)
agent = system.agent(
    name=AGENT_NAME,
    instructions=(
        "Usa inspect_public_api para verificar el simbolo solicitado. "
        "Responde con el nombre, si es publico y la version observada."
    ),
    tools=[inspect_public_api],
    contract=toolkit.AgentContract(
        must_call=["inspect_public_api"],
        completion="when_required_tools_satisfied",
    ),
    policy=toolkit.RunPolicy(
        max_turns=3,
        max_tool_calls=1,
        temperature=0.0,
        trace="compact",
        strict=True,
    ),
)

toolkit.show_json(agent.info(), title="Agente declarado")

## 3) Ejecutar o reportar not-run

La ejecucion live esta habilitada por defecto cuando VLLM_BASE_URL y VLLM_MODEL estan configurados. Usa RUN_VLLM_LIVE=0 para forzar un estado not-run seguro.

In [ ]:
can_run = (
    RUN_VLLM_LIVE
    and vllm_environment.get("base_url_configured")
    and vllm_environment.get("model_configured")
)

if can_run:
    result = agent.run(
        "Verifica si system pertenece a la API publica instalada.",
        mode="eval",
    )
    assert isinstance(result, toolkit.RunResult)
    assert result.ok, result.errors
    assert result.engine == "vllm-runtime"
    toolkit.human_result(result, title="vLLM RunResult", show_lineage=True)
    toolkit.show_json(toolkit.run_result_output(result), title="Contrato normalizado")
else:
    result = None
    toolkit.show_json({
        "status": "not-run",
        "provider": "vllm-runtime",
        "reason": "Configura VLLM_BASE_URL y VLLM_MODEL, o usa RUN_VLLM_LIVE=0.",
    }, title="vLLM live gate")

## 4) API realmente ejercitada

No hay cliente OpenAI ni health check local paralelo: la ejecucion real pertenece al provider de Agentic Systems.

In [ ]:
api_coverage = [
    "toolkit.vllm_environment_snapshot",
    "toolkit.scheduler",
    "toolkit.runtime",
    "toolkit.tool",
    "toolkit.system",
    "system.agent",
    "agent.run",
    "toolkit.human_result",
    "toolkit.run_result_output",
    "toolkit.RunResult",
    "toolkit.show_json",
    "toolkit.AgentContract",
    "toolkit.RunPolicy",
]

toolkit.show_json(api_coverage, title="vLLM API coverage")

## Resultado e interpretacion

Con live desactivado: configuracion observable y estado not-run estructurado. Con live activado: el mismo contrato `RunResult` que OpenAI y Bedrock, cuyo runtime reporta `vllm-runtime`.